# 08 · Query-based detector: RT-DETR

Trains Ultralytics RT-DETR-L on the shared data and scores it with the shared evaluator, next to YOLO (07), Faster R-CNN (02) and RetinaNet (04).
Uses the same helpers as 07 (`common/yolo.py`).

| Section | What it does |
|---|---|
| Setup, protocol | Paths (local, or Kaggle via `kaggle_launch.py`), data config, the training recipe |
| Experiments | `baseline` (300 queries, 640 px), then one change each: 500 queries, learning rate, 960 px |
| Train | `RUN = [...]` cell; `run_queue.py 08_rtdetr.ipynb:baseline,...` runs it unattended |
| Comparison | RT-DETR next to YOLO at the same size (`yolo/imgsz_640`) and the best CNNs; per-epoch curves |
| Inference-only checks, speed, failure analysis, submission | As in 07 |

**How RT-DETR differs from the CNN detectors:** a CNN backbone and a hybrid encoder (attention within the coarsest map, convolutions across maps)
feed a transformer decoder. The decoder starts from a fixed number of **queries** (300), picked as the top-scoring encoder features, and refines them
into boxes. Training matches predictions to objects one-to-one (Hungarian matching), so there is **no NMS** and no anchors;
each image yields at most `num_queries` boxes.

## Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

# Local defaults; kaggle_launch.py sets these variables to the Kaggle input/working folders.
SRC_DIR = Path(os.environ.get("SRC_DIR", Path.cwd()))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "../multiple-object-video-detection-hw-2-ucu-ai-2026"))
PROC_ROOT = Path(os.environ.get("PROC_ROOT", "../data_processed"))
RUNS_ROOT = Path(os.environ.get("RUNS_ROOT", "../runs"))

sys.path.insert(0, str(SRC_DIR))
import ultralytics
from ultralytics import RTDETR

from common.analysis import GroundTruth, error_breakdown, error_table, miss_and_confusion, show_frame
from common.data import CLASSES
from common.engine import load_summaries
from common.evaluate import evaluate_detections
from common.submission import write_submission
from common.yolo import measure_fps, predict_coco, predict_test, train_and_score, val_items, write_data_yaml

FIXTURE = os.environ.get("FIXTURE") == "1"   # tiny CPU smoke test; never report its numbers
GT_FILE = PROC_ROOT / "val" / "gt_eval_coco.json"
TEST_CSV = DATA_ROOT / "test_images.csv"
RUNS_DIR = RUNS_ROOT / ("rtdetr_fixture" if FIXTURE else "rtdetr")
RUNS_DIR.mkdir(parents=True, exist_ok=True)
DATA_YAML = write_data_yaml(PROC_ROOT, RUNS_DIR / "data.yaml")
DEVICE = 0 if torch.cuda.is_available() else "cpu"

# RT-DETR-L with 500 queries: Ultralytics' rtdetr-l.yaml with the decoder's `nq` changed. The query count does not change any
# weight shape (queries are selected encoder features), so every COCO weight still transfers.
_cfg = yaml.safe_load((Path(ultralytics.__file__).parent / "cfg/models/rt-detr/rtdetr-l.yaml").read_text())
_cfg["head"][-1][-1] = ["nc", 256, 500]                # RTDETRDecoder(nc, [ch], hd=256, nq=500); "nc" stays a placeholder
RTDETR_L_500Q = RUNS_DIR / "rtdetr-l-500q.yaml"
RTDETR_L_500Q.write_text(yaml.safe_dump(_cfg, sort_keys=False))

print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "| fixture:", FIXTURE)

## Protocol

Shared with every model: same training frames, original val frames for scoring, the shared evaluator, `conf=0.001`, up to 500 detections, seed 42.
Same YOLO-side choices as 07 (4 epochs, short warm-up, `nbs=16`, mosaic off for the last epoch, no vertical flip or rotation), with these RT-DETR-specific changes:
- **Optimizer AdamW, learning rate 1e-4**, the RT-DETR paper's fine-tuning rate. Ultralytics' `auto` would pick ~8e-4 for a run this short, which is high for a transformer.
- **640 px**, RT-DETR's native size (the decoder samples a fixed number of points per query, and memory grows fast with size). It is compared with YOLO at the same size, `yolo/imgsz_640`.
- **`deterministic=False`:** the deformable attention uses `grid_sample`, which has no deterministic backward pass, so runs are not bit-for-bit reproducible.
- **Mixed precision** is on; Ultralytics warns it can produce NaNs in RT-DETR. `baseline_fp32` is the fallback the Kaggle launcher runs if `baseline` fails.

In [ ]:
PROTOCOL = dict(
    epochs=4,
    batch=8,                   # fixed (16 GB GPU at 640 px); AutoBatch is unreliable for RT-DETR
    train=dict(
        optimizer="AdamW", lr0=1e-4, lrf=0.1, weight_decay=1e-4, seed=42, deterministic=False, amp=True,
        workers=int(os.environ.get("WORKERS", 4)), cache=False,
        fliplr=0.5, flipud=0.0, degrees=0.0, copy_paste=0.0,
        close_mosaic=1, warmup_epochs=0.5, nbs=16,
        save_period=1, plots=False, val=True, device=DEVICE,
    ),
    predict=dict(conf=0.001, iou=0.7, max_det=500, device=DEVICE, batch=4),
    score_every=1,
    val_max_images=None,
)
if FIXTURE:
    PROTOCOL.update(epochs=1, batch=2, val_max_images=6)
    PROTOCOL["train"].update(workers=0, amp=False, close_mosaic=0)
    PROTOCOL["predict"].update(batch=2)

## Experiments

| Name | Change | Question it answers |
|---|---|---|
| `baseline` | `rtdetr-l.pt` (COCO), 300 queries, 640 px | Query-based detector at the same size as `yolo/imgsz_640` |
| `queries_500` | 500 queries | Busy frames hold 150–180 objects (01): do more queries raise recall in dense scenes? |
| `lr_2e-4` | learning rate 2e-4 | Is 1e-4 too slow for a 4-epoch schedule? |
| `imgsz_960` | 960 px (batch 4 to fit memory; updates still every 16 images) | Does resolution help the transformer as much as it helped YOLO? |
| `baseline_fp32` | mixed precision off | Fallback only, if `baseline` hits NaNs |

In [ ]:
BASE = dict(model="rtdetr-l.pt", weights=None, imgsz=640, train={}, predict={})

EXPERIMENTS = {
    "baseline":      {},
    # COCO weights via the trainer's `pretrained` argument: loading them into the yaml model first hits an Ultralytics
    # class-remapping bug for RT-DETR heads
    "queries_500":   dict(model=str(RTDETR_L_500Q), train=dict(pretrained="rtdetr-l.pt")),
    "lr_2e-4":       dict(train=dict(lr0=2e-4)),
    "imgsz_960":     dict(imgsz=960, batch=4),        # 2.25x the pixels: half the batch to fit 16 GB (nbs=16 keeps the update size)
    "baseline_fp32": dict(train=dict(amp=False)),
}
if FIXTURE:
    BASE.update(imgsz=320)

## Train

In [ ]:
RUN = ["baseline"]          # then ["queries_500", "lr_2e-4", "imgsz_960"]
for name in RUN:
    train_and_score(name, {**BASE, **EXPERIMENTS[name]}, PROTOCOL, DATA_YAML, RUNS_DIR, GT_FILE, DATA_ROOT, RTDETR)

## Comparison

RT-DETR runs next to YOLO at the same size (`yolo/imgsz_640`), the best YOLO run, and the best Faster R-CNN and RetinaNet runs.
Copy the Kaggle `runs/rtdetr/` folder into the local `runs/` first.

In [ ]:
detr = load_summaries(RUNS_DIR)
families = {fam: load_summaries(RUNS_ROOT / fam) for fam in ["yolo", "faster_rcnn", "retinanet"]}
rows = {f"rtdetr/{n}": r["best"] for n, r in detr.items()}
picked = {}
for fam, runs in families.items():
    names = set()
    if runs:
        names.add(max(runs, key=lambda n: runs[n]["best"]["mAP"]))
    if fam == "yolo" and "imgsz_640" in runs:
        names.add("imgsz_640")
    for n in sorted(names):
        rows[f"{fam}/{n}"] = runs[n]["best"]
        picked.setdefault(fam, []).append(n)

cols = ["epoch", "mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"] + [f"AP_{c}" for c in CLASSES]
table = pd.DataFrame({n: {c: b.get(c, np.nan) for c in cols} for n, b in rows.items()}).T
table.style.format({c: "{:.4f}" for c in cols[1:]}, na_rep="").highlight_max(axis=0, subset=cols[1:])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
curves = [("rtdetr", n, RUNS_DIR / n) for n in detr] + [(f, n, RUNS_ROOT / f / n) for f, ns in picked.items() for n in ns]
for fam, name, d in curves:
    if not (d / "history.json").exists():
        continue
    h = pd.DataFrame(json.loads((d / "history.json").read_text()))
    style = "-" if fam == "rtdetr" else "--"
    for ax, col in zip(axes, ["mAP", "AP_small", "AR"]):
        ax.plot(h.epoch, h[col], style, marker="o", ms=3, label=f"{fam}/{name}")
for ax, t in zip(axes, ["val mAP@[.5:.95]", "val AP_small", "val AR (500 dets)"]):
    ax.set(xlabel="epoch", ylabel=t, title=t)
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Observations:** _fill in_

## Sanity check: our evaluator vs Ultralytics' metric (grey-filled val, its own matching)

In [ ]:
pd.DataFrame({n: {"ours mAP": r["best"]["mAP"], "ultralytics mAP50-95": r["best"].get("ul_metrics/mAP50-95(B)"),
                  "ours AP50": r["best"]["AP50"], "ultralytics mAP50": r["best"].get("ul_metrics/mAP50(B)")}
              for n, r in detr.items()}).T.round(4)

## Inference-only checks (no training)

On the best RT-DETR run: predicting at a larger size than trained, and how many of the 300 query slots are actually used per frame
(if busy frames fill all of them, the query count caps recall).

In [ ]:
BEST = max(detr, key=lambda n: detr[n]["best"]["mAP"]) if detr else None
print("best RT-DETR run:", BEST)
if BEST:
    cfg = detr[BEST]["config"]
    items = val_items(GT_FILE, DATA_ROOT, PROTOCOL["val_max_images"])
    ids = [i for i, _ in items]
    checks = {}
    for label, kw in {"as trained": {}, "predict at 960": {"imgsz": 320 if FIXTURE else 960}}.items():
        dets = predict_coco(RTDETR(str(RUNS_DIR / BEST / "best.pt")), items,
                            **{"imgsz": cfg["imgsz"], **PROTOCOL["predict"], **cfg.get("predict", {}), **kw})
        checks[label] = evaluate_detections(GT_FILE, dets, image_ids=ids)
    (RUNS_DIR / BEST / "inference_checks.json").write_text(json.dumps(checks, indent=1))
    display(pd.DataFrame(checks).T[["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "AR"]].round(4))

    dets = json.loads((RUNS_DIR / BEST / "val_dets.json").read_text())
    per_img = pd.Series([d["image_id"] for d in dets]).value_counts()
    n_gt = pd.Series({im: len(v) for im, v in GroundTruth(GT_FILE).objects.items()})
    print(f"boxes per frame: median {per_img.median():.0f}, max {per_img.max()}; "
          f"objects per frame: median {n_gt.median():.0f}, max {n_gt.max()}")

**Observations:** _fill in_

## Speed

Images per second at batch size 1 on 50 preloaded val frames (FP32), as in 07.

In [ ]:
items = val_items(GT_FILE, DATA_ROOT)
speed_paths = [p for _, p in items[::max(1, len(items) // 50)][:50]]
pd.Series({f"rtdetr/{n}": measure_fps(RTDETR(str(RUNS_DIR / n / "best.pt")), speed_paths, r["config"]["imgsz"], device=DEVICE)
           for n, r in detr.items()}, name="FPS").round(1)

## Failure analysis

Error types at score ≥ 0.3 for every RT-DETR run and the YOLO / CNN runs picked above, then the best run's worst frames and miss rates.

In [ ]:
SCORE_THR = 0.3
gt = GroundTruth(GT_FILE)
breakdown = {f"rtdetr/{n}": error_breakdown(gt, json.loads((RUNS_DIR / n / "val_dets.json").read_text()), SCORE_THR) for n in detr}
for fam, ns in picked.items():
    for n in ns:
        f = RUNS_ROOT / fam / n / "val_dets.json"
        if f.exists():
            breakdown[f"{fam}/{n}"] = error_breakdown(gt, json.loads(f.read_text()), SCORE_THR)
pd.DataFrame(breakdown).T.round(3)

In [ ]:
dets = json.loads((RUNS_DIR / BEST / "val_dets.json").read_text())
errors = error_table(gt, dets, SCORE_THR)
for col, title in [("missed", "Most missed objects"), ("fp", "Most false positives"), ("wrong_class", "Most class confusions")]:
    worst = errors.sort_values(col, ascending=False).iloc[0]
    show_frame(gt, dets, int(worst.image_id), DATA_ROOT, SCORE_THR, f"{title}: {int(worst[col])} of {int(worst.n_gt)} objects")
miss, confusion = miss_and_confusion(gt, dets, SCORE_THR)
print("Miss rate by class and size:"); display(miss.round(3))
if len(confusion):
    print("Class confusions (true -> predicted):"); display(confusion)

**Observations:** _fill in_

## Test submission

Writes `runs/rtdetr/<SUBMIT>/submission.csv` (classes 0–7, original pixels; RT-DETR returns at most `num_queries` boxes per image).

In [ ]:
SUBMIT = BEST
cfg = detr[SUBMIT]["config"]
preds = predict_test(RTDETR(str(RUNS_DIR / SUBMIT / "best.pt")), TEST_CSV, DATA_ROOT, max_images=2 if FIXTURE else None,
                     **{"imgsz": cfg["imgsz"], **PROTOCOL["predict"], **cfg.get("predict", {})})
sub_path = RUNS_DIR / SUBMIT / "submission.csv"
sub = write_submission(preds, TEST_CSV, sub_path)
n_boxes = sub.PredictionString.fillna("").str.split().str.len().fillna(0) // 6
classes = {int(t) for s in sub.PredictionString.dropna() for t in s.split()[0::6]}
assert classes <= set(range(8)), classes
assert n_boxes.max() <= 500
print(f"Wrote {sub_path}: {len(sub)} rows, {int((n_boxes > 0).sum())} with detections, max {int(n_boxes.max())} boxes per image")